In [2]:
import re

import numpy as np
import pandas as pd
import datetime as dt
import regex

In [14]:

def parseData():

    #handle missing values by hand as it is less cumbersome
    helsinki = pd.read_excel("Helsinki Kaisaniemi.xlsx")
    rollo =  pd.read_excel("Rovaniemi railway.xlsx")
    tampere = pd.read_excel("Tampere Siilinkari.xlsx")




    ### prep for concatting

    for city in [helsinki, rollo, tampere]:
        location = re.search("...",city.iloc[0,0]).group()
        print(location)
        city["Hour"] = city["Time [Local time]"]
        date = (
        city["Day"].astype(str) + "-" +
        city["Month"].astype(str) + "-" +
        city["Year"].astype(str) + " " +
        city["Hour"].astype(str))
        city["TimeStamp"] = pd.to_datetime(date, format="%d-%m-%Y %H:%M") + dt.timedelta(hours=2)

        city.drop(columns=["Time [Local time]", "Observation station", "Day", "Month", "Year", "Hour"], inplace=True)
        city.rename(columns={"Average temperature [°C]": "Mean "+location,
                             "Maximum temperature [°C]" : "Max "+location,
                             "Minimum temperature [°C]" : "Min "+ location},inplace=True)
    cities = helsinki.merge(rollo, how="inner", left_on="TimeStamp", right_on="TimeStamp").merge(tampere, how="inner", left_on="TimeStamp", right_on="TimeStamp")
    #Parse demand
    consumption = pd.read_excel("consumption.xlsx")         #change to finnish time
    consumption["startTime"] = pd.to_datetime(consumption["startTime"])
    date = (consumption["startTime"].dt.day.astype(str) + "-"+
        consumption["startTime"].dt.month.astype(str)+ "-"+
        consumption["startTime"].dt.year.astype(str)+ " "+
        consumption["startTime"].dt.hour.astype(str))
    consumption["TimeStamp"]= pd.to_datetime(date, dayfirst=True)
    consumption = consumption.iloc[:,2:4]
    consumption = consumption.groupby(["TimeStamp"]).mean()
    df = cities.merge(consumption, how="inner", left_on="TimeStamp", right_on="TimeStamp")
    df.set_index("TimeStamp", inplace=True)
    return df

def writeCSV(name="parsittuData.csv"):
    df = parseData()
    df.to_csv(name)
    print(f"CSV file " + name +" saved!")

writeCSV()

Hel
Rov
Tam
CSV file parsittuData.csv saved!


Hel
Rov
Tam
